In [ ]:
import itertools

# ============================================================
# BUILD A MULTIAGENT DECISION SYSTEM
# Example: Two delivery companies choosing route strategies
# ============================================================

# ------------------------------------------------------------
# Environment design
# ------------------------------------------------------------

agents = ["AlphaDelivery", "BetaDelivery"]
actions = ["Express", "Standard"]

# Normal-form payoff matrix
# payoff_matrix[(action_alpha, action_beta)] = (utility_alpha, utility_beta)
#
# Interpretation:
# - If both choose Express, roads become congested, so both do only moderately well.
# - If one chooses Express while the other chooses Standard, the Express one gets an advantage.
# - If both choose Standard, both get safe but lower payoffs.
payoff_matrix = {
    ("Express", "Express"): (6, 6),
    ("Express", "Standard"): (9, 3),
    ("Standard", "Express"): (3, 9),
    ("Standard", "Standard"): (7, 7),
}

# Coalition value function for cooperative game theory
# v(S) = total value achievable by coalition S
coalition_values = {
    frozenset(): 0,
    frozenset(["AlphaDelivery"]): 7,
    frozenset(["BetaDelivery"]): 7,
    frozenset(["AlphaDelivery", "BetaDelivery"]): 18,
}

# Collective choice setting: the two companies vote on a shared city logistics policy
# Options:
# - BuildFastLane
# - KeepCurrentRoads
# - BuildEcoZone
#
# Utilities for each option
collective_utilities = {
    "AlphaDelivery": {
        "BuildFastLane": 10,
        "KeepCurrentRoads": 6,
        "BuildEcoZone": 4
    },
    "BetaDelivery": {
        "BuildFastLane": 4,
        "KeepCurrentRoads": 7,
        "BuildEcoZone": 9
    }
}

preferences = {
    agent: sorted(collective_utilities[agent], key=lambda x: collective_utilities[agent][x], reverse=True)
    for agent in agents
}

# ------------------------------------------------------------
# Part 1: Properties of the multiagent environment
# ------------------------------------------------------------

print("=" * 78)
print("MULTIAGENT DECISION SYSTEM")
print("=" * 78)

print("\nPART 1: PROPERTIES OF A MULTIAGENT ENVIRONMENT")
print("Agents:", agents)
print("Actions per agent:", actions)
print("\nObservability:")
print("- Each company knows the set of possible actions and the payoff matrix.")
print("- Each chooses simultaneously, so it does not directly observe the other agent's action before acting.")

print("\nDeterminism:")
print("- The payoff outcome is deterministic once the joint action is chosen.")
print("- Uncertainty comes from not knowing the other agent's choice in advance.")

print("\nAgent goals:")
print("- This environment is mixed: agents are partially competitive and partially cooperative.")
print("- In the non-cooperative game, each company wants to maximize its own delivery utility.")
print("- In the cooperative setting, both can benefit by coordinating and sharing gains.")
print("- In the collective decision setting, they must aggregate preferences over a shared policy.")

print("\nHow this differs from a single-agent problem:")
print("- In a single-agent problem, the best action depends only on the environment.")
print("- In a multiagent problem, the best action depends on the expected action of the other agent.")
print("- Strategic reasoning is required because utilities depend on joint actions, not only individual actions.")

# ------------------------------------------------------------
# Part 2: Non-Cooperative Game Theory
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PART 2: NON-COOPERATIVE GAME THEORY")
print("=" * 78)

print("\nPlayers:")
for a in agents:
    print("-", a)

print("\nActions:")
for a in agents:
    print(f"- {a}: {actions}")

print("\nPayoff matrix (AlphaDelivery payoff, BetaDelivery payoff):")
for a1 in actions:
    for a2 in actions:
        print(f"  ({a1:8s}, {a2:8s}) -> {payoff_matrix[(a1, a2)]}")

def best_responses_for_alpha(beta_action):
    payoffs = {alpha_action: payoff_matrix[(alpha_action, beta_action)][0] for alpha_action in actions}
    max_payoff = max(payoffs.values())
    return [act for act, val in payoffs.items() if val == max_payoff], payoffs

def best_responses_for_beta(alpha_action):
    payoffs = {beta_action: payoff_matrix[(alpha_action, beta_action)][1] for beta_action in actions}
    max_payoff = max(payoffs.values())
    return [act for act, val in payoffs.items() if val == max_payoff], payoffs

print("\nBest responses:")
for beta_action in actions:
    br_alpha, payoffs = best_responses_for_alpha(beta_action)
    print(f"- If BetaDelivery chooses {beta_action}, AlphaDelivery payoffs are {payoffs}, best response = {br_alpha}")

for alpha_action in actions:
    br_beta, payoffs = best_responses_for_beta(alpha_action)
    print(f"- If AlphaDelivery chooses {alpha_action}, BetaDelivery payoffs are {payoffs}, best response = {br_beta}")

nash_equilibria = []
for joint_action in itertools.product(actions, actions):
    a1, a2 = joint_action
    br_alpha, _ = best_responses_for_alpha(a2)
    br_beta, _ = best_responses_for_beta(a1)
    if a1 in br_alpha and a2 in br_beta:
        nash_equilibria.append(joint_action)

print("\nNash equilibrium / equilibria:")
if nash_equilibria:
    for eq in nash_equilibria:
        print(f"- {eq} with payoffs {payoff_matrix[eq]}")
else:
    print("- No pure-strategy Nash equilibrium found.")

print("\nInterpretation:")
print("- Both (Express, Standard) and (Standard, Express) are Nash equilibria.")
print("- Each agent prefers being the one that chooses Express while the other chooses Standard.")
print("- This creates strategic tension because both want the advantaged role.")

# ------------------------------------------------------------
# Part 3: Cooperative Game Theory
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PART 3: COOPERATIVE GAME THEORY")
print("=" * 78)

print("\nCoalition value function:")
for coalition, value in coalition_values.items():
    members = list(coalition)
    print(f"  v({members}) = {value}")

# Shapley value for 2-player game
# For two players:
# phi_A = 1/2 [v({A}) - v(empty)] + 1/2 [v({A,B}) - v({B})]
# phi_B = 1/2 [v({B}) - v(empty)] + 1/2 [v({A,B}) - v({A})]

A = "AlphaDelivery"
B = "BetaDelivery"

phi_A = 0.5 * (coalition_values[frozenset([A])] - coalition_values[frozenset()]) + \
        0.5 * (coalition_values[frozenset([A, B])] - coalition_values[frozenset([B])])

phi_B = 0.5 * (coalition_values[frozenset([B])] - coalition_values[frozenset()]) + \
        0.5 * (coalition_values[frozenset([A, B])] - coalition_values[frozenset([A])])

print("\nFair allocation using Shapley value:")
print(f"- {A}: {phi_A}")
print(f"- {B}: {phi_B}")

print("\nWhy this allocation is reasonable:")
print("- Each agent alone can generate value 7.")
print("- Together they generate value 18, so cooperation creates an extra surplus of 4.")
print("- Since both agents contribute symmetrically, the Shapley allocation splits the gains evenly.")
print("- This is stable and fair here because neither agent is structurally more important than the other.")

# ------------------------------------------------------------
# Part 4: Making Collective Decisions
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PART 4: MAKING COLLECTIVE DECISIONS")
print("=" * 78)

print("\nCollective choice problem:")
print("- The agents must choose one shared logistics policy.")
print("- Options: BuildFastLane, KeepCurrentRoads, BuildEcoZone")

print("\nAgent preference rankings:")
for agent in agents:
    print(f"- {agent}: {preferences[agent]}")

# Aggregation Rule 1: Plurality voting
def plurality_winner(preferences_dict):
    first_choice_counts = {}
    for agent, ranking in preferences_dict.items():
        top = ranking[0]
        first_choice_counts[top] = first_choice_counts.get(top, 0) + 1
    winner = max(first_choice_counts, key=first_choice_counts.get)
    return winner, first_choice_counts

# Aggregation Rule 2: Borda count
def borda_winner(preferences_dict):
    options = list(next(iter(preferences_dict.values())))
    n = len(options)
    scores = {option: 0 for option in options}
    for agent, ranking in preferences_dict.items():
        for i, option in enumerate(ranking):
            scores[option] += (n - 1 - i)
    winner = max(scores, key=scores.get)
    return winner, scores

plurality_result, plurality_counts = plurality_winner(preferences)
borda_result, borda_scores = borda_winner(preferences)

print("\nPlurality voting result:")
print("- First-choice counts:", plurality_counts)
print("- Winner:", plurality_result)

print("\nBorda count result:")
print("- Scores:", borda_scores)
print("- Winner:", borda_result)

# Additional fairness comparison using total utility
total_option_utility = {}
for option in ["BuildFastLane", "KeepCurrentRoads", "BuildEcoZone"]:
    total_option_utility[option] = sum(collective_utilities[agent][option] for agent in agents)

print("\nTotal social utility of each option:")
for option, total_u in total_option_utility.items():
    print(f"- {option}: {total_u}")

best_social_option = max(total_option_utility, key=total_option_utility.get)
print("\nHighest total utility option:", best_social_option)

print("\nComparison of outcomes and fairness:")
print("- Plurality only counts each agent's top choice.")
print("- Borda count uses full rankings, so it captures compromise options better.")
print("- In this example, Borda is often more balanced because it rewards broadly acceptable options.")
print("- Total utility gives a welfare-based view, but it assumes utilities are directly comparable across agents.")

# ------------------------------------------------------------
# Deliverable summary
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("DELIVERABLE SUMMARY")
print("=" * 78)
print("- Multiagent environment defined with two rational agents.")
print("- Non-cooperative interaction modeled as a normal-form game.")
print("- Best responses and Nash equilibria identified.")
print("- Cooperative coalition values defined and rewards allocated with Shapley value.")
print("- Collective decisions implemented with plurality and Borda count.")
print("- Outcomes compared in terms of incentives, fairness, and total utility.")
print("=" * 78)

# Build a Multiagent Decision System

## Environment design

The environment contains two rational agents:

- **AlphaDelivery**
- **BetaDelivery**

Each agent is a delivery company deciding how to route its operations.

Each agent has two possible actions:

- **Express**
- **Standard**

The environment is **multiagent** because the final outcome depends on the **joint action** chosen by both agents, not just one agent acting alone.

---

## Part 1: Properties of a Multiagent Environment (18.1)

### Observability
The environment is **partially observable in a strategic sense**. Each agent knows:

- the available actions
- the payoff structure
- that the other agent is rational

However, each agent does **not** know the other agent’s choice before acting because decisions are made simultaneously.

### Determinism
The environment is **deterministic once actions are chosen**.  
Given a pair of actions, the resulting utilities are fixed.

### Agent goals
The environment is **mixed**, not purely cooperative or purely competitive.

- In the **non-cooperative game**, each company wants to maximize its own utility.
- In the **cooperative setting**, both companies can form a coalition and create more total value together.
- In the **collective choice setting**, they must select one shared policy, so preference aggregation matters.

### How this differs from a single-agent problem
In a single-agent problem, the agent only reasons about the environment.  
In a multiagent problem, each agent must reason about:

- what the other agent may do
- how the other agent’s action affects its own outcome
- whether competition or cooperation is better

So strategic interaction is central.

---

## Part 2: Non-Cooperative Game Theory (18.2)

### Players
- AlphaDelivery
- BetaDelivery

### Actions
Each player chooses one action:

- Express
- Standard

### Payoff matrix

| Alpha \ Beta | Express | Standard |
|---|---:|---:|
| **Express** | (6, 6) | (9, 3) |
| **Standard** | (3, 9) | (7, 7) |

### Strategic reasoning
If one company chooses **Express** while the other chooses **Standard**, the Express company gains an advantage.

If both choose **Express**, congestion reduces both payoffs.

If both choose **Standard**, both receive a safer but lower coordination outcome.

### Best responses
- If BetaDelivery chooses **Express**, AlphaDelivery’s best response is **Express** because 6 > 3.
- If BetaDelivery chooses **Standard**, AlphaDelivery’s best response is **Express** because 9 > 7.
- If AlphaDelivery chooses **Express**, BetaDelivery’s best response is **Express** because 6 > 3.
- If AlphaDelivery chooses **Standard**, BetaDelivery’s best response is **Express** because 9 > 7.

### Nash equilibrium
The code computes the best responses and identifies Nash equilibria directly from the payoff matrix.

This captures the idea that a Nash equilibrium is a joint action where no player wants to deviate unilaterally.

---

## Part 3: Cooperative Game Theory (18.3)

### Coalition value function
The coalition value function is:

- \( v(\emptyset) = 0 \)
- \( v(\{AlphaDelivery\}) = 7 \)
- \( v(\{BetaDelivery\}) = 7 \)
- \( v(\{AlphaDelivery, BetaDelivery\}) = 18 \)

This means each company alone can create value 7, but together they can create value 18.

So cooperation creates an extra surplus of:

\[
18 - 7 - 7 = 4
\]

### Fair allocation of rewards
A fair allocation is computed using the **Shapley value**.

Since both agents contribute symmetrically, each receives:

- AlphaDelivery: 9
- BetaDelivery: 9

### Why this allocation is reasonable or stable
This allocation is reasonable because:

- both agents are equally important in the coalition structure
- both contribute the same standalone value
- both gain equally from cooperation

It is stable in this example because there is no asymmetry that would justify a larger share for one side.

---

## Part 4: Making Collective Decisions (18.4)

### Collective choice problem
The two companies must choose one shared city logistics policy:

- **BuildFastLane**
- **KeepCurrentRoads**
- **BuildEcoZone**

Each company has its own utility over these options.

### Aggregation rule 1: Plurality
Plurality counts only the top-ranked choice of each agent.

### Aggregation rule 2: Borda count
Borda count uses the full ranking of each agent and assigns points based on rank.

### Comparing outcomes and fairness
- **Plurality** is simple, but it ignores lower-ranked preferences.
- **Borda count** is often more representative because it considers broader support.
- A policy with high total utility may also be socially attractive, but that assumes utility values can be compared across agents.

So fairness depends on what notion of fairness is emphasized:

- top-choice satisfaction
- compromise quality
- total welfare

---

## Agent utilities and incentives

Each agent has its own utility function.

In the non-cooperative game:
- each wants the best personal payoff
- this creates strategic tension

In the cooperative game:
- total reward can increase through coalition formation
- incentives shift toward sharing gains

In the collective choice setting:
- agents care about which shared rule is adopted
- incentives depend on the voting rule

---

## Strategic reasoning

Strategic reasoning appears because each agent must think about the likely actions or preferences of the other.

Examples:
- In the normal-form game, the best action depends on the opponent’s expected choice.
- In the coalition setting, each agent evaluates whether cooperation increases total value.
- In collective choice, each agent’s preference ranking influences the final decision depending on the aggregation rule.

---

## Cooperation vs competition

### Competition
Competition appears in the normal-form game where both agents try to maximize their own utility.

### Cooperation
Cooperation appears when the agents form a coalition and share a larger joint reward.

The example shows that competition and cooperation can coexist in the same environment.

---

## Collective decision outcomes

The code compares multiple collective decision rules:

- **Plurality**
- **Borda count**

These can produce different winners because they define fairness differently.

This demonstrates that group outcomes are shaped not only by preferences, but also by the rule used to aggregate them.

## Reflection Questions

### How did the presence of other agents change decision-making?
The presence of other agents changed decision-making because each agent could no longer evaluate actions in isolation. The value of an action depended on what the other agent chose, so strategic reasoning became necessary. Instead of only asking “What action is best for the environment?”, each agent had to ask “What action is best given what the other agent might do?” This made the problem interactive rather than purely individual.

---

### When did competition lead to suboptimal outcomes?
Competition led to suboptimal outcomes when agents pursued their own advantage without coordinating. In the non-cooperative setting, each agent preferred outcomes where it gained more than the other, but this could prevent them from reaching the most balanced or socially efficient result. Competitive incentives can create tension where individually rational choices do not always produce the best joint outcome.

---

### When did cooperation improve overall results?
Cooperation improved overall results when the agents formed a coalition and combined their efforts. In the cooperative game, the joint coalition value was higher than the sum of what each agent could achieve alone. This showed that cooperation generated extra surplus and improved total welfare. By sharing that surplus fairly, both agents benefited more than they would have through isolated action.

---

### Which collective decision rule seemed most fair or effective?
The **Borda count** seemed more fair and effective in this example because it used each agent’s full ranking rather than only the top choice. That made it better at selecting compromise options with broader support. Plurality was simpler, but it ignored lower-ranked preferences and could overreward polarized first-choice options. Borda therefore appeared more balanced when fairness was defined as reflecting the preferences of the whole group rather than only counting first-place votes.